## Tutorial on integrating ReaktoroBlock into NF unit model on a flowsheet
Demonstration of how to use Reaktoro block with a unit model from WaterTAP

#### Dependencies
* Python - Programming language
* Pyomo - Python package for equation-oriented modeling
* IDAES - Python package extending Pyomo for flowsheet modeling
* cyipopt - Solver necessary for use with gray box models
* Reaktoro-pse - Python package for building Reaktoro gray box models
* watertap-solvers - for cyipopt solver wrapper
* WaterTAP - Unit models

### Demonstration structure 
* Setting up basic speciation block and calculating properties for a feed composition
    * How to adjust apparent species to achieve thermodynamic equilibrium
* Adding Reaktoro block to a [NF WaterTAP](https://watertap.readthedocs.io/en/latest/technical_reference/unit_models/nanofiltration_ZO.html) model for calculation of local osmotic pressure and scaling tendency


### Example for speciation and use with NF model

### Import needed modules

In [1]:
## Import core components
# Pyomo core components
from pyomo.environ import (
    Var,
    Constraint,
    TransformationFactory,
    Reals,
    ConcreteModel,
    value,
    assert_optimal_termination,
    units as pyunits,
)
from pyomo.network import Arc

# Ideas core components
from idaes.core import FlowsheetBlock
from idaes.core.util.scaling import (
    calculate_scaling_factors,
    set_scaling_factor,
    constraint_scaling_transform,
)

from idaes.core.util.model_statistics import degrees_of_freedom, unfixed_variables_in_activated_equalities_set, activated_equalities_set

from idaes.core.util.initialization import propagate_state

from idaes.models.unit_models import Feed
from pyomo.util.calc_var_value import calculate_variable_from_constraint

import watertap.property_models.multicomp_aq_sol_prop_pack as props 
# Import reaktoro-pse and reaktoro
from reaktoro_pse.reaktoro_block import ReaktoroBlock
import reaktoro
from reaktoro_pse.core.util_classes.cyipopt_solver import (
    get_cyipopt_watertap_solver,
)

### Incorporating Reaktoro to calculate Seawater properties in NF Desalination
Sea water desalination is a common process. Here it is demonstrated how ReaktoroBlock can be used:
 1) Calculate feed density, achieve charge neutrality, and estimate amount of acid need to operate at target pH
 2) Calculate osmotic pressure in feed and retentate of NF ZO model

In [2]:
# This a typical composition of sea water with ion concentration in mg/L and pH

sea_water_composition = {
    "Na": 10556,
    "K": 380,
    "Ca": 400,
    "Mg": 1262,
    "Cl": 17000,
    "SO4": 2649,
    "HCO3": 140,
}
sea_water_ph = 7.56
sea_water_TDS = sum(sea_water_composition.values())

### Define standard Pyomo model and WaterTAP property package

In [3]:
import json

with open('solute_parameters.json') as f:
    solute_data = json.load(f)

# Generate solute data from JSON file
solute_list = list(solute_data.keys())
mw_data = {key: solute_data[key]['mw'] for key in solute_list}
charge = {key: solute_data[key]['charge'] for key in solute_list}
diffusivity = {("Liq", key): 1e-9 for key in solute_list}

m = ConcreteModel()
# create IDAES flowsheet
m.fs = FlowsheetBlock(dynamic=False)

# Initialize MCAS properties
m.fs.properties = props.MCASParameterBlock(
    solute_list=solute_list,
    mw_data=mw_data,
    charge=charge,
    diffusivity_data=diffusivity,
    density_calculation=props.DensityCalculation.seawater,
    material_flow_basis=props.MaterialFlowBasis.mass
)
# build feed
m.fs.feed = Feed(property_package=m.fs.properties)

### Define inputs for Reaktoro and outputs
MCAS tracks species but separate variable is created to track species mass flow for Reaktoro

In [4]:
# Get ions
ions = list(sea_water_composition.keys())

# ReaktoroBlock needs mass flows of all the species
ions.append("H2O")
m.fs.feed.species_mass_flow = Var(
    ions, initialize=1, bounds=(0, None), units=pyunits.kg / pyunits.s
)

# Create pH variable for feed
m.fs.feed.pH = Var(initialize=sea_water_ph)
m.fs.feed.pH.fix()

# The solution is not charge neutralized, so to get true TDS, we will need to adjust concentration of one of the ions"""
m.fs.feed.reaktoro_charge = Var(initialize=1e-8, bounds=(None, None))

# To get true flow mass, the density is needed for the solution, we can get that from reaktoro
m.fs.feed.reaktoro_density = Var(
    initialize=1000, units=pyunits.kg / pyunits.m**3
)  # default density unit returned by reaktoro - https://reaktoro.org/api/classReaktoro_1_1ChemicalProps.html#afd97c7985258fa29f5a69634c07a0ac7

# We can also get osmotic pressure to compare to NaCl prop pack predictions, and pH after acid addition
m.fs.feed.reaktoro_osmotic_pressure = Var(initialize=1, units=pyunits.Pa)

### Writing constraints for calculation of mass flows

In [5]:
# Write constraints to convert concentration to mass flows
@m.fs.feed.Constraint(list(m.fs.feed.species_mass_flow.keys()))
def eq_feed_species_mass_flow(fs, ion):
    if ion == "H2O":
        return (
            m.fs.feed.species_mass_flow["H2O"]
            == m.fs.feed.properties[0].flow_mass_phase_comp["Liq", "H2O"]
        )
    else:
        return m.fs.feed.species_mass_flow[ion] == pyunits.convert(
            m.fs.feed.properties[0].flow_mass_phase_comp["Liq", ion],
            to_units=pyunits.kg / pyunits.s,
        )

for ion in m.fs.feed.species_mass_flow.keys():
    print(m.fs.feed.species_mass_flow[ion])



    

fs.feed.species_mass_flow[Na]
fs.feed.species_mass_flow[K]
fs.feed.species_mass_flow[Ca]
fs.feed.species_mass_flow[Mg]
fs.feed.species_mass_flow[Cl]
fs.feed.species_mass_flow[SO4]
fs.feed.species_mass_flow[HCO3]
fs.feed.species_mass_flow[H2O]


### Reaktoro Block output dict

In [6]:
"""We need to define an output dictionary with our properties - this can also be an Pyomo IndexedVar please check examples in examples folder for how to use IndexedVars as outputs"""

m.fs.feed.reaktoro_outputs = {
    (
        "osmoticPressure",
        "H2O",
    ): m.fs.feed.reaktoro_osmotic_pressure,  # not how the second key is the water, we can get osmotic pressure for different components in the system
    ("density", None): m.fs.feed.reaktoro_density,
    ("charge", None): m.fs.feed.reaktoro_charge,
}  # - this will force reaktor to return exact speciation with all species

### Configure ReaktoroBlock
We are dealing with seawater desalination, which implies operation at high salinity and pressure for such conditions the PhreeqC Pitzer data base is a good choice 
For comparison of PhreeqC data base following paper is a good reference: https://doi.org/10.1016/j.earscirev.2021.103888 

In [7]:
m.fs.feed.charge_neutrality = ReaktoroBlock(
    aqueous_phase={
        "composition": m.fs.feed.species_mass_flow,  # This is the spices mass flow
        "convert_to_rkt_species": True,  # We can use default converter as its defined for default database (Phreeqc and pitzer)
        "activity_model": reaktoro.ActivityModelPitzer(),  # Can provide a string, or Reaktoro initialized class
        "fixed_solvent_specie": "H2O",  # We need to define our aqueous solvent as we have to speciate the block
    },
    system_state={
        "temperature": m.fs.feed.properties[0].temperature,
        "pressure": m.fs.feed.properties[0].pressure,
        "pH": m.fs.feed.pH,
    },
    outputs=m.fs.feed.reaktoro_outputs,  # This is the output dictionary we defined above
    database_file="pitzer.dat",  # needs to be a string that names the database file or points to its location
    dissolve_species_in_reaktoro=True,  # This will sum up all species into elements in Reaktoro directly, if set to false, it will build Pyomo constraints instead
    assert_charge_neutrality=False,  # This is True by Default, but here we actually want to adjust the input speciation till the charge is zero
    build_speciation_block=False,  # We provided apparent species so we need to speciate them.
)

2026-04-09 21:38:46 [INFO] idaes.reaktoro_pse.core.reaktoro_inputs: Exact speciation is not provided! Fixing aqueous solvent and, excluding H
2026-04-09 21:38:46 [INFO] idaes.reaktoro_pse.core.reaktoro_inputs: Exact speciation is not provided! Fixing aqueous solvent and, excluding O
2026-04-09 21:38:47 [INFO] idaes.reaktoro_pse.core.reaktoro_solver: rktSolver inputs: ['[C]', '[Na]', '[Mg]', '[S]', '[Cl]', '[K]', '[Ca]', '[H2O]', '[H]', '[O]', '[H+]']
2026-04-09 21:38:47 [INFO] idaes.reaktoro_pse.core.reaktoro_solver: rktSolver constraints: ['C_constraint', 'Na_constraint', 'Mg_constraint', 'S_constraint', 'Cl_constraint', 'K_constraint', 'Ca_constraint', 'H2O_constraint', 'H_dummy_constraint', 'O_dummy_constraint', 'pH']
2026-04-09 21:38:47 [INFO] idaes.reaktoro_pse.core.reaktoro_gray_box: RKT gray box using LBFGS hessian type


Lets inspect outputs from main reaktoro model

Note how we are missing osmoticPressure, and instead have speciesActivityLn and speciesStandardVolume as our outputs, this is becouse we osmoticPressure is a pyomo property, rather a native property supplied by reaktoro. You can inspect how this property is created by checking the:
* osmoticPressure in PyomoProperties class located in reaktoro_pse.core.reaktoro_outputs
* build_osmotic_constraint in reaktoro_pse.core.pyomo_property_writer.property_functions

This two location will also show how we can access reaktoro database to pull out fixed parameters and create a custom pyomo property. 

### Set default values for feed and scaling

In [8]:
m.fs.feed.properties[0].temperature.fix(273 + 25)  # temperature (K)
m.fs.feed.properties[0].pressure.fix(101325)  # pressure (Pa)
m.fs.feed.properties[0].flow_mass_phase_comp["Liq", "H2O"].fix(
    0.965
)  # mass flowrate of H2O (kg/s)
m.fs.feed.properties[0].conc_mass_phase_comp[...]  # construct concentration props
m.fs.feed.properties[0].pressure_osm_phase[...]
m.fs.properties.set_default_scaling(
    "flow_mass_phase_comp",
    1 / 0.965,
    index=("Liq", "H2O"),
)

for key, value in sea_water_composition.items():
    m.fs.properties.set_default_scaling(
        "flow_mass_phase_comp",
        1 / (value / 1000000),  # aproximage scale
        index=("Liq", key),
    )

### Initialize our composition constraints and scale all the variables

In [9]:
for ion, value in sea_water_composition.items():
    ion_conc = value / 1000 # in kg/m**3
    mass_flow_ion = ion_conc * m.fs.feed.properties[0].flow_mass_phase_comp["Liq", "H2O"].value / m.fs.feed.reaktoro_density.value
    m.fs.feed.properties[0].flow_mass_phase_comp["Liq", ion].fix(mass_flow_ion)
    m.fs.feed.properties[0].conc_mass_phase_comp["Liq", ion].unfix()
    set_scaling_factor(m.fs.feed.properties[0].flow_mass_phase_comp["Liq", ion], 1 / mass_flow_ion)


for comp, pyoobj in m.fs.feed.eq_feed_species_mass_flow.items():
    calculate_variable_from_constraint(m.fs.feed.species_mass_flow[comp], pyoobj)
    set_scaling_factor(
        m.fs.feed.species_mass_flow[ion], 1 / m.fs.feed.species_mass_flow[comp].value
    )
    constraint_scaling_transform(pyoobj, 1 / m.fs.feed.species_mass_flow[comp].value)

set_scaling_factor(m.fs.feed.reaktoro_density, 1 / 1000)
set_scaling_factor(m.fs.feed.reaktoro_osmotic_pressure, 1 / 1e5)
set_scaling_factor(m.fs.feed.pH, 1)
set_scaling_factor(m.fs.feed.reaktoro_charge, 1e4)

### Intialize feed and reaktoro block. 
Reaktoro initialization does several steps:

1) Initialize input constraints propagating them from user variables to Reaktoro graybox inputs
2) Solve the Reaktoro block to get output properties 
3) Propagate Reaktoro solution through output constraints and to output variables 
4) Scale all input and output variables and constraints using either user provided scaling factors or by inverse of their value 
5) Scale the jacobian using user provided scaling or inverse of scaling factors of the gray box outputs

This will in general provide a well scaled problem. 

In [10]:
m.fs.feed.initialize()
m.fs.feed.charge_neutrality.initialize()
# lets solve the model with current state:
print("DOFs:", degrees_of_freedom(m))
cy_solver = get_cyipopt_watertap_solver(max_iter=150)
result = cy_solver.solve(m, tee=True)
assert_optimal_termination(result)

2026-04-09 21:38:47 [INFO] idaes.init.fs.feed: Initialization Complete.
2026-04-09 21:38:47 [INFO] idaes.reaktoro_pse.reaktoro_block: ---initializing property block fs.feed.charge_neutrality----
2026-04-09 21:38:47 [INFO] idaes.reaktoro_pse.core.reaktoro_state: Equilibrated successfully
2026-04-09 21:38:48 [WARNING] idaes.reaktoro_pse.core.reaktoro_block_builder: Jacobian scale for ('osmoticPressure', 'H2O') below 1e-08, set to 1e-08
2026-04-09 21:38:48 [INFO] idaes.reaktoro_pse.core.reaktoro_block_builder: Initialized rkt block
DOFs: 0
cyipopt-watertap: cyipopt with user variable scaling and IDAES jacobian constraint scaling

List of user-set options:

                                    Name   Value                used
              acceptable_constr_viol_tol = 1e-09                 yes
                 acceptable_dual_inf_tol = 0.01                  yes
                          acceptable_tol = 1e-09                 yes
                      bound_relax_factor = 0                  

In [11]:
# these are manually defined properties and variables of interest
print(
    "Density reaktoro",
    m.fs.feed.reaktoro_density.value,
    f"Density MCAS",
    m.fs.feed.properties[0].dens_mass_phase["Liq"].value,
)
print(
    "Osmotic pressure",
    m.fs.feed.reaktoro_osmotic_pressure.value,
    f"Osmotic pressure MCAS",
    m.fs.feed.properties[0].pressure_osm_phase["Liq"].value,
)
print("Solution reaktoro_charge", m.fs.feed.reaktoro_charge.value)

Density reaktoro 1022.0748714005878 Density MCAS 1020.8392240913253
Osmotic pressure 2306850.487115441 Osmotic pressure MCAS 2564867.272736861
Solution reaktoro_charge 0.05384365602377509


### Lets solve the current model to:
* Find actual mass flows of species 
* Solution density
* Required Cl amount to get zero charge in solution

In [12]:
# unfix Cl and fix charge to 0

m.fs.feed.species_mass_flow["Cl"].unfix()
m.fs.feed.properties[0].flow_mass_phase_comp["Liq", "Cl"].unfix()
m.fs.feed.reaktoro_charge.fix(0)

Lets check DOFs before solve, and note that its equal to number of our reaktoro outputs

In [13]:
initial_cl = m.fs.feed.species_mass_flow["Cl"].value
print("DOFs:", degrees_of_freedom(m))
assert degrees_of_freedom(m) == 0
result = cy_solver.solve(m, tee=True)
assert_optimal_termination(result)

DOFs: 0
cyipopt-watertap: cyipopt with user variable scaling and IDAES jacobian constraint scaling

List of user-set options:

                                    Name   Value                used
              acceptable_constr_viol_tol = 1e-09                 yes
                 acceptable_dual_inf_tol = 0.01                  yes
                          acceptable_tol = 1e-09                 yes
                      bound_relax_factor = 0                     yes
                         constr_viol_tol = 1e-08                 yes
                  diverging_iterates_tol = 1e+30                 yes
                            dual_inf_tol = 0.1                   yes
                   honor_original_bounds = no                    yes
                           linear_solver = mumps                 yes
                                max_iter = 150                   yes
                      nlp_scaling_method = user-scaling          yes
                      print_user_options = ye

In [14]:
print(
    "Density reaktoro",
    m.fs.feed.reaktoro_density.value,
    f"Density MCAS",
    m.fs.feed.properties[0].dens_mass_phase["Liq"].value,
)
print(
    "Osmotic pressure",
    m.fs.feed.reaktoro_osmotic_pressure.value,
    f"Density MCAS",
    m.fs.feed.properties[0].pressure_osm_phase["Liq"].value,
)
print(
    "Solution reaktoro_charge",
    m.fs.feed.reaktoro_charge.value,
    "intial Cl",
    initial_cl,
    "final Cl",
    m.fs.feed.properties[0].flow_mass_phase_comp["Liq", "Cl"].value,
)

Density reaktoro 1023.0138430194143 Density MCAS 1022.251021027268
Osmotic pressure 2440371.7051867642 Density MCAS 2701893.794765991
Solution reaktoro_charge 0 intial Cl 0.016405 final Cl 0.018313839093275802


### Adding [WaterTAP NF model](https://watertap.readthedocs.io/en/latest/technical_reference/unit_models/nanofiltration_ZO.html) and replacing default osmotic pressure with reaktoro calculations.
Import WaterTAP NF ZO model and pump, and build them.


In [15]:
from watertap.unit_models.nanofiltration_ZO import NanofiltrationZO
from watertap.unit_models.pressure_changer import Pump

m.fs.pump = Pump(property_package=m.fs.properties)
m.fs.NF = NanofiltrationZO(property_package=m.fs.properties)

# connect feed to pump
m.fs.feed_to_pump = Arc(source=m.fs.feed.outlet, destination=m.fs.pump.inlet)
# connect pump to NF unit
m.fs.pump_to_nf = Arc(source=m.fs.pump.outlet, destination=m.fs.NF.inlet)
# Expand arcs
TransformationFactory("network.expand_arcs").apply_to(m)

of the Nanofiltration0D model. The Nanofiltation0D model has most of the
capabilities of the NanofiltrationZO model with additional support for
ensuring electroneutrality in outlets.  (deprecated in 1.3.0) (called from
d:\Software\miniconda\envs\reaktoro-pse-dev\Lib\site-
packages\idaes\core\base\process_block.py:128)


Add reaktoro blocks for osmotic pressure calculations. This is done for both feed and retentate.

Here we will pack together osmotic pressure, scaling tendency and pH as outputs together to be returned by ReaktoroBlock for both streams. 

Lets check that our outputs are correctly packaged

In [16]:
locations = ["feed", "retentate"]



m.fs.NF.scaling_tendency = Var(
    locations,
    (("scalingTendency", "Calcite"), ("scalingTendency", "Gypsum")),
    initialize=1,
)
set_scaling_factor(m.fs.NF.scaling_tendency, 1)

m.fs.NF.species_mass_flow = Var(
    locations,
    list(m.fs.feed.species_mass_flow.keys()),
    initialize=1,
    units=pyunits.kg / pyunits.s,
    domain=Reals,
)

@m.fs.Constraint(list(m.fs.NF.species_mass_flow.keys()))
def eq_ro_interphase_flow_mass_comp(fs, loc, ion):
    if loc == "feed":
        if ion == "H2O":  # flow of water is same
            return (
                m.fs.NF.species_mass_flow[loc, "H2O"]
                == m.fs.NF.feed_side.properties_in[0].flow_mass_phase_comp[
                    "Liq", "H2O"
                ]
            )
        else:
            return (
                m.fs.NF.species_mass_flow[loc, ion]
                == m.fs.NF.feed_side.properties_in[0].flow_mass_phase_comp[
                    "Liq", ion
                ]
            )
    if loc == "retentate":
        if ion == "H2O":  # flow of water is same
            return (
                m.fs.NF.species_mass_flow[loc, "H2O"]
                == m.fs.NF.feed_side.properties_out[0].flow_mass_phase_comp[
                    "Liq", "H2O"
                ]
            )
        else:
            return (
                m.fs.NF.species_mass_flow[loc, ion]
                == m.fs.NF.feed_side.properties_out[0].flow_mass_phase_comp[
                    "Liq", ion
                ]
            )
    
m.fs.NF.pH = Var(
    locations,
    [("pH", None)],
    initialize=1,
)
set_scaling_factor(m.fs.NF.pH, 1)

m.fs.NF.reaktoro_outputs = {}

m.fs.NF.reaktoro_outputs[("feed", "osmoticPressure", "H2O")] = m.fs.NF.feed_side.properties_in[0].pressure_osm_phase["Liq"]
m.fs.NF.reaktoro_outputs[("retentate", "osmoticPressure", "H2O")] = m.fs.NF.feed_side.properties_out[0].pressure_osm_phase["Liq"]
for idx, obj in m.fs.NF.scaling_tendency.items():
    m.fs.NF.reaktoro_outputs[idx] = obj
for idx, obj in m.fs.NF.pH.items():
    m.fs.NF.reaktoro_outputs[idx] = obj

m.fs.NF_pressure = {}
m.fs.NF_pressure["feed"] = m.fs.NF.feed_side.properties_in[0].pressure
m.fs.NF_pressure["retentate"] = m.fs.NF.feed_side.properties_out[0].pressure

In [17]:
for key, obj in m.fs.NF.reaktoro_outputs.items():
    print(key, obj)

('feed', 'osmoticPressure', 'H2O') fs.NF.feed_side.properties_in[0.0].pressure_osm_phase[Liq]
('retentate', 'osmoticPressure', 'H2O') fs.NF.feed_side.properties_out[0.0].pressure_osm_phase[Liq]
('feed', 'scalingTendency', 'Calcite') fs.NF.scaling_tendency[feed,scalingTendency,Calcite]
('feed', 'scalingTendency', 'Gypsum') fs.NF.scaling_tendency[feed,scalingTendency,Gypsum]
('retentate', 'scalingTendency', 'Calcite') fs.NF.scaling_tendency[retentate,scalingTendency,Calcite]
('retentate', 'scalingTendency', 'Gypsum') fs.NF.scaling_tendency[retentate,scalingTendency,Gypsum]
('feed', 'pH', None) fs.NF.pH[feed,pH,None]
('retentate', 'pH', None) fs.NF.pH[retentate,pH,None]


Finally, lets build all the reaktoro blocks.

In [18]:
m.fs.eq_nf_chem_props = ReaktoroBlock(
    locations,
    aqueous_phase={
        "composition": m.fs.NF.species_mass_flow,  # This is the spices mass flow
        "convert_to_rkt_species": True,  # We can use default converter as its defined for default database (Phreeqc and pitzer)
        "activity_model": reaktoro.ActivityModelPitzer(),  # Can provide a string, or Reaktoro initialized class
        "fixed_solvent_specie": "H2O",  # We need to define our aqueous solvent as we have to speciate the block
    },
    system_state={
        "temperature": m.fs.NF.feed_side.properties_in[0].temperature,
        "temperature_indexed": False,
        "pressure": m.fs.NF_pressure,
        "pH": m.fs.feed.pH,
        "pH_indexed": False,  # we are not providing unique pH at each node, so lets disable indexing for it
    },
    outputs=m.fs.NF.reaktoro_outputs,  # outputs we desired
    database="PhreeqcDatabase",  # Can provide a string, or Reaktoro initialized class reaktor.PhreeqcDatabase()
    database_file="pitzer.dat",  # needs to be a string that names the database file or points to its location
    dissolve_species_in_reaktoro=True,  # This will sum up all species into elements in Reaktoro directly, if set to false, it will build Pyomo constraints instead
    assert_charge_neutrality=True,
    build_speciation_block=False,
)

2026-04-09 21:38:49 [INFO] idaes.reaktoro_pse.core.reaktoro_inputs: Exact speciation is not provided! Fixing aqueous solvent and, excluding H
2026-04-09 21:38:49 [INFO] idaes.reaktoro_pse.core.reaktoro_inputs: Exact speciation is not provided! Fixing aqueous solvent and, excluding O
2026-04-09 21:38:50 [INFO] idaes.reaktoro_pse.core.reaktoro_solver: rktSolver inputs: ['[Cl]', '[C]', '[Na]', '[Mg]', '[S]', '[K]', '[Ca]', '[H2O]', '[H]', '[O]', '[H+]']
2026-04-09 21:38:50 [INFO] idaes.reaktoro_pse.core.reaktoro_solver: rktSolver constraints: ['C_constraint', 'Na_constraint', 'Mg_constraint', 'S_constraint', 'K_constraint', 'Ca_constraint', 'H2O_constraint', 'H_dummy_constraint', 'O_dummy_constraint', 'charge', 'pH']
2026-04-09 21:38:50 [INFO] idaes.reaktoro_pse.core.reaktoro_gray_box: RKT gray box using LBFGS hessian type
2026-04-09 21:38:50 [INFO] idaes.reaktoro_pse.core.reaktoro_inputs: Exact speciation is not provided! Fixing aqueous solvent and, excluding H
2026-04-09 21:38:50 [INFO]

Configure NF defaults and scale the model

In [19]:
m.fs.feed.properties[0].pressure_osm_phase[...]
# define pump defaults
m.fs.pump.efficiency_pump[0].fix(0.75)
# scale work and pressures for the pump
set_scaling_factor(m.fs.pump.control_volume.work, 1e-4)
set_scaling_factor(m.fs.pump.control_volume.properties_out[0].pressure, 1e-5)
set_scaling_factor(m.fs.pump.control_volume.properties_in[0].pressure, 1e-5)

# define NF default values for initialization
# we opt to specify area

m.fs.NF.area.fix(100)
set_scaling_factor(m.fs.NF.area, 1)

# we need to specify NF permeate pressure
m.fs.NF.properties_permeate[0].pressure.fix(101325)

# we also specify rejection values for all ions other than Cl

for key in solute_list:
    if key != "Cl":
        m.fs.NF.rejection_phase_comp[0, "Liq", key].fix(solute_data[key]['rejection_phase_comp'])
        
# For Cl we introduce an initial guess and then use this value to assure electroneutrality
m.fs.NF.rejection_phase_comp[0, "Liq", "Cl"] = 0.15

charge_comp = {key: solute_data[key]['charge'] for key in solute_list}

m.fs.NF.eq_electroneutrality = Constraint(
    expr=0
    == sum(
        charge_comp[j]
        * m.fs.NF.properties_permeate[0].conc_mol_phase_comp["Liq", j]
        for j in charge_comp
    )
)
constraint_scaling_transform(m.fs.NF.eq_electroneutrality, 1)
# calculate all the scaling factors
calculate_scaling_factors(m)

Initialize NF model and pump. We fix recovery here.

In [20]:
propagate_state(m.fs.feed_to_pump)
# get osmotic pressure
osmotic_feed_pressure = m.fs.feed.properties[0].pressure_osm_phase["Liq"].value
print("Osmotic pressure is {} bar".format(osmotic_feed_pressure / 1e5))
m.fs.pump.outlet.pressure[0].fix(osmotic_feed_pressure * 1.5)
m.fs.pump.initialize()

propagate_state(m.fs.pump_to_nf)
m.fs.NF.recovery_vol_phase.fix(0.5)
m.fs.NF.initialize()

Osmotic pressure is 27.01893794765991 bar
2026-04-09 21:38:51 [INFO] idaes.init.fs.pump.control_volume: Initialization Complete
component keys that are not exported as part of the NL file.  Skipping.
that are not Var, Constraint, Objective, or the model.  Skipping.
2026-04-09 21:38:51 [INFO] idaes.init.fs.pump: Initialization Complete: optimal - Optimal Solution Found
2026-04-09 21:38:51 [INFO] idaes.init.fs.NF.feed_side: Initialization Complete
2026-04-09 21:38:51 [INFO] idaes.init.fs.NF.feed_side.properties_in: fs.NF.feed_side.properties_in State Released.
2026-04-09 21:38:51 [INFO] idaes.init.fs.NF: Initialization Complete: optimal - Optimal Solution Found


Initialize water removal and deactivate NF osmotic pressure constraints

In [21]:
for (loc, ion), obj in m.fs.NF.species_mass_flow.items():
    calculate_variable_from_constraint(
        m.fs.NF.species_mass_flow[loc, ion],
        m.fs.eq_ro_interphase_flow_mass_comp[loc, ion],
    )
    sf = 1 / m.fs.feed.species_mass_flow[ion].value
    set_scaling_factor(m.fs.NF.species_mass_flow[loc, ion], sf)
    constraint_scaling_transform(m.fs.eq_ro_interphase_flow_mass_comp[loc, ion], sf)
m.fs.NF.feed_side.properties_in[0].eq_pressure_osm_phase[
    "Liq"
].deactivate()
m.fs.NF.feed_side.properties_out[0].eq_pressure_osm_phase[
    "Liq"
].deactivate()

Initialize all the ReaktoroBlocks on NF model, we have to iterate over them as they are indexed blocks

In [22]:
for blk, obj in m.fs.eq_nf_chem_props.items():
    obj.initialize()

2026-04-09 21:38:51 [INFO] idaes.reaktoro_pse.reaktoro_block: ---initializing property block fs.eq_nf_chem_props[feed]----
2026-04-09 21:38:51 [INFO] idaes.reaktoro_pse.core.reaktoro_state: Equilibrated successfully
2026-04-09 21:38:52 [WARNING] idaes.reaktoro_pse.core.reaktoro_block_builder: Jacobian scale for ('osmoticPressure', 'H2O') below 1e-08, set to 1e-08
2026-04-09 21:38:52 [INFO] idaes.reaktoro_pse.core.reaktoro_block_builder: Initialized rkt block
2026-04-09 21:38:52 [INFO] idaes.reaktoro_pse.reaktoro_block: ---initializing property block fs.eq_nf_chem_props[retentate]----
2026-04-09 21:38:52 [INFO] idaes.reaktoro_pse.core.reaktoro_state: Equilibrated successfully
2026-04-09 21:38:52 [WARNING] idaes.reaktoro_pse.core.reaktoro_block_builder: Jacobian scale for ('osmoticPressure', 'H2O') below 1e-08, set to 1e-08
2026-04-09 21:38:52 [INFO] idaes.reaktoro_pse.core.reaktoro_block_builder: Initialized rkt block


Lets check our outputs make sense

In [23]:
for key, obj in m.fs.NF.reaktoro_outputs.items():
    print(key, obj.value)

('feed', 'osmoticPressure', 'H2O') 2445732.5119460877
('retentate', 'osmoticPressure', 'H2O') 2768621.190995344
('feed', 'scalingTendency', 'Calcite') 1.1075511836763332
('feed', 'scalingTendency', 'Gypsum') 0.19152251111583948
('retentate', 'scalingTendency', 'Calcite') 2.876363403478726
('retentate', 'scalingTendency', 'Gypsum') 0.43868636364599667
('feed', 'pH', None) 7.560000000001508
('retentate', 'pH', None) 7.560000000003461


Solve NF model with new osmotic pressure calculation 

In [24]:
result = cy_solver.solve(m, tee=True)
assert_optimal_termination(result)

cyipopt-watertap: cyipopt with user variable scaling and IDAES jacobian constraint scaling

List of user-set options:

                                    Name   Value                used
              acceptable_constr_viol_tol = 1e-09                 yes
                 acceptable_dual_inf_tol = 0.01                  yes
                          acceptable_tol = 1e-09                 yes
                      bound_relax_factor = 0                     yes
                         constr_viol_tol = 1e-08                 yes
                  diverging_iterates_tol = 1e+30                 yes
                            dual_inf_tol = 0.1                   yes
                   honor_original_bounds = no                    yes
                           linear_solver = mumps                 yes
                                max_iter = 150                   yes
                      nlp_scaling_method = user-scaling          yes
                      print_user_options = yes       

# Compare osmotic pressure for NaCl vs Reaktoro

In [25]:
reaktoro_osm = []
mcas_prop_pack_osm = []

for loc in locations:
    reaktoro_osm.append(
        float(m.fs.NF.reaktoro_outputs[loc, "osmoticPressure", "H2O"].value / 1e5)
    )
    """ recalc pressure from prop pack package"""
    if loc == "retentate":
        calculate_variable_from_constraint(
            m.fs.NF.reaktoro_outputs[loc, "osmoticPressure", "H2O"],
            m.fs.NF.feed_side.properties_out[0].eq_pressure_osm_phase["Liq"],
        )
    elif loc == "feed":
        calculate_variable_from_constraint(
            m.fs.NF.reaktoro_outputs[loc, "osmoticPressure", "H2O"],
            m.fs.NF.feed_side.properties_in[0].eq_pressure_osm_phase["Liq"],
        )
    mcas_prop_pack_osm.append(
        float(m.fs.NF.reaktoro_outputs[loc, "osmoticPressure", "H2O"].value / 1e5)
    )

print(f"Reaktoro osmotic pressures are {reaktoro_osm[0]} at inlet and {reaktoro_osm[1]} at outlet")
print(f"MCAS osmotic pressures are {mcas_prop_pack_osm[0]} at inlet and {mcas_prop_pack_osm[1]} at outlet")

Reaktoro osmotic pressures are 24.457325119461217 at inlet and 27.68621254727055 at outlet
MCAS osmotic pressures are 27.018937947516815 at inlet and 31.061129846262236 at outlet


Check other properties

In [26]:

scaling_calcite = []
scaling_gypsum = []
for loc in locations:
    scaling_calcite.append(
        float(m.fs.NF.reaktoro_outputs[loc, "scalingTendency", "Calcite"].value)
    )
    scaling_gypsum.append(
        float(m.fs.NF.reaktoro_outputs[loc, "scalingTendency", "Gypsum"].value)
    )
print(f"Calcite scaling tendency is {scaling_calcite[0]} at inlet and {scaling_calcite[1]} at outlet")
print(f"Gypsum scaling tendency is {scaling_gypsum[0]} at inlet and {scaling_gypsum[1]} at outlet")

Calcite scaling tendency is 1.1075511836832055 at inlet and 2.876363984071779 at outlet
Gypsum scaling tendency is 0.19152251111583596 at inlet and 0.4386864401298382 at outlet


Solve NF for additional recoveries. 

In [27]:
print("Current recovery", m.fs.NF.recovery_vol_phase[0, "Liq"].value)
print("Current Cl removal", m.fs.NF.rejection_phase_comp[0, "Liq", "Cl"].value)
m.fs.NF.recovery_vol_phase[0, "Liq"].fix(0.6)
m.fs.pump.outlet.pressure[0].unfix()
result = cy_solver.solve(m, tee=True)
assert_optimal_termination(result)

Current recovery 0.5
Current Cl removal 0.14628236536404354
cyipopt-watertap: cyipopt with user variable scaling and IDAES jacobian constraint scaling

List of user-set options:

                                    Name   Value                used
              acceptable_constr_viol_tol = 1e-09                 yes
                 acceptable_dual_inf_tol = 0.01                  yes
                          acceptable_tol = 1e-09                 yes
                      bound_relax_factor = 0                     yes
                         constr_viol_tol = 1e-08                 yes
                  diverging_iterates_tol = 1e+30                 yes
                            dual_inf_tol = 0.1                   yes
                   honor_original_bounds = no                    yes
                           linear_solver = mumps                 yes
                                max_iter = 150                   yes
                      nlp_scaling_method = user-scaling       

Show comparison at higher recovery

In [28]:
reaktoro_osm = []
mcas_prop_pack_osm = []
for loc in locations:
    reaktoro_osm.append(
        float(m.fs.NF.reaktoro_outputs[loc, "osmoticPressure", "H2O"].value / 1e5)
    )
    """ recalc pressure from prop pack package"""
    if loc == "retentate":
        calculate_variable_from_constraint(
            m.fs.NF.reaktoro_outputs[loc, "osmoticPressure", "H2O"],
            m.fs.NF.feed_side.properties_out[0].eq_pressure_osm_phase["Liq"],
        )
    elif loc == "feed":
        calculate_variable_from_constraint(
            m.fs.NF.reaktoro_outputs[loc, "osmoticPressure", "H2O"],
            m.fs.NF.feed_side.properties_in[0].eq_pressure_osm_phase["Liq"],
        )
    mcas_prop_pack_osm.append(
        float(m.fs.NF.reaktoro_outputs[loc, "osmoticPressure", "H2O"].value / 1e5)
    )



scaling_calcite = []
scaling_gypsum = []
for loc in locations:
    scaling_calcite.append(
        float(m.fs.NF.reaktoro_outputs[loc, "scalingTendency", "Calcite"].value)
    )
    scaling_gypsum.append(
        float(m.fs.NF.reaktoro_outputs[loc, "scalingTendency", "Gypsum"].value)
    )
print(f"Reaktoro osmotic pressures are {reaktoro_osm[0]} at inlet and {reaktoro_osm[1]} at outlet")
print(f"MCAS osmotic pressures are {mcas_prop_pack_osm[0]} at inlet and {mcas_prop_pack_osm[1]} at outlet")
print(f"Calcite scaling tendency is {scaling_calcite[0]} at inlet and {scaling_calcite[1]} at outlet")
print(f"Gypsum scaling tendency is {scaling_gypsum[0]} at inlet and {scaling_gypsum[1]} at outlet")
print("Current Cl removal ", m.fs.NF.rejection_phase_comp[0, "Liq", "Cl"].value)

Reaktoro osmotic pressures are 25.20280544717996 at inlet and 30.268319558871603 at outlet
MCAS osmotic pressures are 27.01893794760864 at inlet and 33.08157045976526 at outlet
Calcite scaling tendency is 0.5453473993167975 at inlet and 1.944824191646399 at outlet
Gypsum scaling tendency is 0.08287985611072723 at inlet and 0.24922401788935014 at outlet
Current Cl removal  0.14628236536404354
